# Chandelier Exit with Entry Filter on SPY
## Strategy Brief
The Chandelier Exit is a trend-following strategy that uses the Average True Range (ATR) to set trailing stop-loss levels. This strategy incorporates an entry filter to time entries more effectively by ensuring that trades are only initiated when the price is above a certain moving average, indicating an uptrend. The strategy aims to capture significant price movements while minimizing losses during sideways markets. Backtesting on SPY, the strategy shows potential for capturing long-term trends with improved risk management.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for our trading strategy including the ATR period, multiplier, and moving average period for the entry filter.

In [ ]:
ATR_PERIOD = 22
ATR_MULTIPLIER = 3.0
MA_PERIOD = 50
START_DATE = '2010-01-01'

## PHASE 2 - Data Exploration
We will download SPY data from Yahoo Finance, calculate the Chandelier Exit indicator, and plot it alongside the price to visualize potential entry and exit points.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE)

# Calculate ATR
high_low = data['High'] - data['Low']
high_close = np.abs(data['High'] - data['Close'].shift())
low_close = np.abs(data['Low'] - data['Close'].shift())
tr = high_low.combine(high_close, max).combine(low_close, max)
atr = tr.rolling(ATR_PERIOD).mean()

# Calculate Chandelier Exit
chandelier_exit = data['High'].rolling(ATR_PERIOD).max() - (atr * ATR_MULTIPLIER)

# Calculate Moving Average
moving_average = data['Close'].rolling(MA_PERIOD).mean()

# Plot
plt.figure(figsize=(14,7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(chandelier_exit, label='Chandelier Exit', linestyle='--')
plt.plot(moving_average, label=f'{MA_PERIOD}-day MA', linestyle=':')
plt.title('SPY with Chandelier Exit and Moving Average')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We define the entry and exit logic based on the Chandelier Exit and the moving average filter. The signal is generated when the price is above the moving average and the close price crosses below the Chandelier Exit.

In [ ]:
# Generate signals
signal = (data['Close'] > moving_average) & (data['Close'] < chandelier_exit)

# Entry and exit logic
positions = pd.Series(np.where(signal, -1, 0), index=data.index)

## PHASE 4 - Coding & Backtesting
We backtest the strategy by calculating daily returns and plotting the equity curve to visualize the strategy's performance over time.

In [ ]:
# Shift positions to align with next day returns
positions = positions.shift(1).fillna(0)

# Calculate daily returns
returns = data['Close'].pct_change().fillna(0)
strategy_returns = positions * returns

# Compute equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14,7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('Equity Curve of Chandelier Exit Strategy')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We evaluate the strategy's performance using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We also compare these metrics to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (equity_curve[-1] ** (252 / len(returns))) - 1
    sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(252)
    downside_returns = returns[returns < 0]
    sortino_ratio = np.mean(returns) / np.std(downside_returns) * np.sqrt(252)
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

# Calculate metrics for strategy
strategy_metrics = calculate_performance_metrics(strategy_returns)

# Calculate metrics for buy-and-hold
buy_hold_returns = data['Close'].pct_change().fillna(0)
buy_hold_equity_curve = (1 + buy_hold_returns).cumprod()
buy_hold_metrics = calculate_performance_metrics(buy_hold_returns)

# Display comparison table
comparison_table = pd.DataFrame(
    [strategy_metrics, buy_hold_metrics],
    columns=['CAGR', 'Sharpe', 'Sortino', 'Calmar', 'Max Drawdown'],
    index=['Strategy', 'Buy-and-Hold']
)
print(comparison_table)

## PHASE 6 - Deploy & Monitor
We create a function to download the last 60 days of SPY data, compute today's signal, and print the recommended position based on the strategy.

In [ ]:
def get_latest_signal():
    # Download last 60 days of data
    recent_data = yf.download('SPY', period='60d')
    
    # Recalculate indicators
    recent_tr = recent_data['High'] - recent_data['Low']
    recent_high_close = np.abs(recent_data['High'] - recent_data['Close'].shift())
    recent_low_close = np.abs(recent_data['Low'] - recent_data['Close'].shift())
    recent_tr = recent_tr.combine(recent_high_close, max).combine(recent_low_close, max)
    recent_atr = recent_tr.rolling(ATR_PERIOD).mean()
    recent_chandelier_exit = recent_data['High'].rolling(ATR_PERIOD).max() - (recent_atr * ATR_MULTIPLIER)
    recent_moving_average = recent_data['Close'].rolling(MA_PERIOD).mean()
    
    # Compute today's signal
    today_signal = (recent_data['Close'][-1] > recent_moving_average[-1]) & (recent_data['Close'][-1] < recent_chandelier_exit[-1])
    position = -1 if today_signal else 0
    print(f"Today's position: {'Short' if position == -1 else 'Flat'}")

get_latest_signal()